In [8]:
import json

# Load both result files
baseline = {}
hyde = {}
hyde_docs = {}

with open('all_results/scifact/bge_baseline/per_query_results.jsonl') as f:
    for line in f:
        r = json.loads(line)
        baseline[r['query_id']] = r

with open('all_results/scifact/bge_hyde/per_query_results.jsonl') as f:
    for line in f:
        r = json.loads(line)
        hyde[r['query_id']] = r

with open('all_results/scifact/contriever_hyde/hyde_docs.jsonl') as f:
    for line in f:
        r = json.loads(line)
        hyde_docs[r['query_id']] = r

In [9]:
# Categorize queries
hyde_helped = []  # HyDE hit@10=1, baseline hit@10=0
hyde_hurt = []    # baseline hit@10=1, HyDE hit@10=0
both_hit = []     # both hit@10=1
both_miss = []    # both hit@10=0
hyde_helped_rank = []  # both hit but HyDE has better rank (hit@1)

for qid in baseline:
    b = baseline[qid]
    h = hyde[qid]
    b10 = b['hit@10']
    h10 = h['hit@10']
    
    if h10 == 1 and b10 == 0:
        hyde_helped.append(qid)
    elif b10 == 1 and h10 == 0:
        hyde_hurt.append(qid)
    elif b10 == 1 and h10 == 1:
        both_hit.append(qid)
    else:
        both_miss.append(qid)

# Also check hit@1 differences among both_hit
hyde_better_top1 = []
baseline_better_top1 = []
for qid in both_hit:
    b1 = baseline[qid].get('hit@1', 0)
    # hyde file uses hit@1 key
    h1 = hyde[qid].get('hit@1', 0)
    if h1 == 1 and b1 == 0:
        hyde_better_top1.append(qid)
    elif b1 == 1 and h1 == 0:
        baseline_better_top1.append(qid)

print("=" * 60)
print("SCIFACT ERROR ANALYSIS: BGE_baseline vs BGE_HyDE (hit@10)")
print("=" * 60)
print(f"Total queries: {len(baseline)}")
print(f"Both hit@10:      {len(both_hit)} ({100*len(both_hit)/300:.1f}%)")
print(f"Both miss@10:     {len(both_miss)} ({100*len(both_miss)/300:.1f}%)")
print(f"HyDE helped:      {len(hyde_helped)} ({100*len(hyde_helped)/300:.1f}%)")
print(f"HyDE hurt:        {len(hyde_hurt)} ({100*len(hyde_hurt)/300:.1f}%)")
print()
print("Among both-hit queries (rank quality at top-1):")
print(f"  HyDE got top-1, baseline didn't: {len(hyde_better_top1)}")
print(f"  Baseline got top-1, HyDE didn't: {len(baseline_better_top1)}")
print()

# Show HyDE-hurt examples with generated docs
print("=" * 60)
print("FAILURE CASES: HyDE HURT (baseline hit@10 but HyDE missed)")
print("=" * 60)
for qid in hyde_hurt:
    b = baseline[qid]
    h = hyde[qid]
    hd = hyde_docs.get(qid, {})
    print(f"\nQuery ID: {qid}")
    print(f"  Claim: {b['query']}")
    print(f"  Gold docs: {b['gold_doc_ids']}")
    print(f"  Baseline hit@1={b['hit@1']}, hit@10={b['hit@10']}")
    print(f"  HyDE hit@1={h['hit@1']}, hit@10={h['hit@10']}")
    hyde_text = hd.get('hyde_doc', h.get('hyde_document', 'N/A'))
    print(f"  HyDE doc: {hyde_text[:200]}...")
    # Check where gold doc appears in baseline
    gold = b['gold_doc_ids'][0]
    if gold in b['retrieved_doc_ids']:
        rank = b['retrieved_doc_ids'].index(gold) + 1
        print(f"  Gold doc rank in baseline: {rank}")
    print(f"  Gold doc rank in HyDE: {h['retrieved_doc_ids'].index(gold) + 1 if gold in h['retrieved_doc_ids'] else 'not found'}")


SCIFACT ERROR ANALYSIS: BGE_baseline vs BGE_HyDE (hit@10)
Total queries: 300
Both hit@10:      252 (84.0%)
Both miss@10:     25 (8.3%)
HyDE helped:      15 (5.0%)
HyDE hurt:        8 (2.7%)

Among both-hit queries (rank quality at top-1):
  HyDE got top-1, baseline didn't: 23
  Baseline got top-1, HyDE didn't: 26

FAILURE CASES: HyDE HURT (baseline hit@10 but HyDE missed)

Query ID: 99
  Claim: Alizarin forms hydrogen bonds with residues involved in PGAM1 substrate binding.
  Gold docs: ['18810195']
  Baseline hit@1=1, hit@10=1
  HyDE hit@1=0, hit@10=0
  HyDE doc: Alizarin, a natural anthraquinone dye, has been shown to interact with various biomolecules through non-covalent interactions, including hydrogen bonding. In the context of phosphoglycerate mutase 1 (...
  Gold doc rank in baseline: 1
  Gold doc rank in HyDE: 18

Query ID: 415
  Claim: Female carriers of the Apolipoprotein E4 (APOE4) allele have increased risk for dementia.
  Gold docs: ['6309659']
  Baseline hit@1=0, hit@10=

In [10]:
# Look at the HyDE-hurt cases more carefully - check the generated docs
hyde_hurt = [qid for qid in baseline 
             if baseline[qid]['hit@10'] == 1 and hyde[qid]['hit@10'] == 0]

print("=" * 60)
print("DETAILED FAILURE ANALYSIS: HyDE-HURT CASES")
print("=" * 60)
for qid in hyde_hurt:
    b = baseline[qid]
    h = hyde[qid]
    hd = hyde_docs.get(qid, {})
    hyde_text = hd.get('hyde_doc', h.get('hyde_document', 'N/A'))
    gold = b['gold_doc_ids'][0]
    b_rank = b['retrieved_doc_ids'].index(gold) + 1 if gold in b['retrieved_doc_ids'] else 'not found'
    h_rank = h['retrieved_doc_ids'].index(gold) + 1 if gold in h['retrieved_doc_ids'] else 'not found'
    
    print(f"\n{'─'*60}")
    print(f"Query {qid}: {b['query']}")
    print(f"  Baseline rank: {b_rank} | HyDE rank: {h_rank}")
    print(f"  HyDE doc (full): {hyde_text}")
    print()

# Also check: among both-miss, how many have negation/contradiction language?
both_miss = [qid for qid in baseline 
             if hyde[qid]['hit@10'] == 0 and baseline[qid]['hit@10'] == 0]

print("\n" + "=" * 60)
print(f"PATTERN ANALYSIS")
print("=" * 60)

# Check for negation words in queries
negation_words = ['not', 'no ', 'without', 'lack', 'reduces', 'impaired', 'poorly', 'less', 'decreased', 'inhibits']
negation_queries_miss = []
negation_queries_all = []
for qid in baseline:
    q = baseline[qid]['query'].lower()
    has_neg = any(w in q for w in negation_words)
    if has_neg:
        negation_queries_all.append(qid)
        if baseline[qid]['hit@10'] == 0 and hyde[qid]['hit@10'] == 0:
            negation_queries_miss.append(qid)

print(f"\nQueries with negation/contradiction language: {len(negation_queries_all)}")
print(f"  Of those, both miss@10: {len(negation_queries_miss)} ({100*len(negation_queries_miss)/max(len(negation_queries_all),1):.1f}%)")
print(f"  Overall both miss rate: {100*len(both_miss)/300:.1f}%")

# Check query length patterns
import statistics
all_lengths = [len(baseline[qid]['query'].split()) for qid in baseline]
miss_lengths = [len(baseline[qid]['query'].split()) for qid in both_miss]
hurt_lengths = [len(baseline[qid]['query'].split()) for qid in hyde_hurt]
helped_lengths = [len(baseline[qid]['query'].split()) for qid in 
                  [q for q in baseline if hyde[q]['hit@10']==1 and baseline[q]['hit@10']==0]]

print(f"\nQuery length (words):")
print(f"  All queries: mean={statistics.mean(all_lengths):.1f}, median={statistics.median(all_lengths):.1f}")
print(f"  Both miss:   mean={statistics.mean(miss_lengths):.1f}, median={statistics.median(miss_lengths):.1f}")
if hurt_lengths:
    print(f"  HyDE hurt:   mean={statistics.mean(hurt_lengths):.1f}, median={statistics.median(hurt_lengths):.1f}")
if helped_lengths:
    print(f"  HyDE helped: mean={statistics.mean(helped_lengths):.1f}, median={statistics.median(helped_lengths):.1f}")

# Number of gold docs per query
multi_gold = [qid for qid in baseline if len(baseline[qid]['gold_doc_ids']) > 1]
multi_gold_miss = [qid for qid in both_miss if len(baseline[qid]['gold_doc_ids']) > 1]
print(f"\nMulti-gold-doc queries: {len(multi_gold)}")
print(f"  Of those, both miss: {len(multi_gold_miss)}")

DETAILED FAILURE ANALYSIS: HyDE-HURT CASES

────────────────────────────────────────────────────────────
Query 99: Alizarin forms hydrogen bonds with residues involved in PGAM1 substrate binding.
  Baseline rank: 1 | HyDE rank: 18
  HyDE doc (full): Alizarin, a natural anthraquinone dye, has been shown to interact with various biomolecules through non-covalent interactions, including hydrogen bonding. In the context of phosphoglycerate mutase 1 (PGAM1), a key enzyme in glycolysis, the binding of alizarin to specific residues can influence the enzyme's activity. Studies indicate that alizarin can form hydrogen bonds with amino acid side chains in the active site of PGAM1, particularly with residues such as histidine and aspartate, which are crucial for substrate binding and catalysis. These interactions can stabilize the enzyme-substrate complex,


────────────────────────────────────────────────────────────
Query 415: Female carriers of the Apolipoprotein E4 (APOE4) allele have increas

In [11]:
# HyDE helped cases
hyde_helped = [qid for qid in baseline 
               if hyde[qid]['hit@10'] == 1 and baseline[qid]['hit@10'] == 0]

print("=" * 60)
print("SUCCESS CASES: HyDE HELPED (baseline miss, HyDE hit@10)")
print("=" * 60)
for qid in hyde_helped:
    b = baseline[qid]
    h = hyde[qid]
    hd = hyde_docs.get(qid, {})
    print(f"\nQuery ID: {qid}")
    print(f"  Claim: {b['query']}")
    print(f"  Gold docs: {b['gold_doc_ids']}")
    print(f"  Baseline hit@1={b['hit@1']}, hit@10={b['hit@10']}")
    print(f"  HyDE hit@1={h['hit@1']}, hit@10={h['hit@10']}")
    hyde_text = hd.get('hyde_doc', h.get('hyde_document', 'N/A'))
    print(f"  HyDE doc: {hyde_text[:200]}...")
    gold = b['gold_doc_ids'][0]
    if gold in h['retrieved_doc_ids']:
        rank = h['retrieved_doc_ids'].index(gold) + 1
        print(f"  Gold doc rank in HyDE: {rank}")
        print(f"  Gold doc rank in Baseline: {b['retrieved_doc_ids'].index(gold) + 1 if gold in b['retrieved_doc_ids'] else 'not found'}")

print()
print("=" * 60)
print("BOTH MISS CASES (neither hit@10)")
print("=" * 60)
both_miss = [qid for qid in baseline 
             if hyde[qid]['hit@10'] == 0 and baseline[qid]['hit@10'] == 0]

for qid in both_miss[:8]:
    b = baseline[qid]
    h = hyde[qid]
    hd = hyde_docs.get(qid, {})
    print(f"\nQuery ID: {qid}")
    print(f"  Claim: {b['query']}")
    print(f"  Gold docs: {b['gold_doc_ids']}")
    hyde_text = hd.get('hyde_doc', h.get('hyde_document', 'N/A'))
    print(f"  HyDE doc: {hyde_text[:150]}...")
    gold = b['gold_doc_ids'][0]
    b_has = gold in b['retrieved_doc_ids']
    h_has = gold in h['retrieved_doc_ids']
    if b_has:
        print(f"  Gold in baseline top-100: rank {b['retrieved_doc_ids'].index(gold)+1}")
    else:
        print(f"  Gold in baseline top-100: NO")
    if h_has:
        print(f"  Gold in HyDE top-100: rank {h['retrieved_doc_ids'].index(gold)+1}")
    else:
        print(f"  Gold in HyDE top-100: NO")


SUCCESS CASES: HyDE HELPED (baseline miss, HyDE hit@10)

Query ID: 1
  Claim: 0-dimensional biomaterials show inductive properties.
  Gold docs: ['31715818']
  Baseline hit@1=0, hit@10=0
  HyDE hit@1=0, hit@10=1
  HyDE doc: 0-dimensional biomaterials, often referred to as nanoparticles, exhibit unique inductive properties due to their high surface area-to-volume ratio and quantum effects. These materials can influence bi...
  Gold doc rank in HyDE: 2
  Gold doc rank in Baseline: 13

Query ID: 303
  Claim: DMRT1 is a sex-determining gene that is epigenetically regulated by the MHM region.
  Gold docs: ['4388470']
  Baseline hit@1=0, hit@10=0
  HyDE hit@1=0, hit@10=1
  HyDE doc: DMRT1 (Doublesex and Mab-3 Related Transcription Factor 1) is a critical gene involved in the determination of male sex in various vertebrate species. It functions as a transcription factor that influ...
  Gold doc rank in HyDE: 6
  Gold doc rank in Baseline: 17

Query ID: 324
  Claim: Deleting Raptor reduces G-C

In [12]:
# Look at the HyDE-hurt cases more carefully - check the generated docs
hyde_hurt = [qid for qid in baseline 
             if baseline[qid]['hit@10'] == 1 and hyde[qid]['hit@10'] == 0]

print("=" * 60)
print("DETAILED FAILURE ANALYSIS: HyDE-HURT CASES")
print("=" * 60)
for qid in hyde_hurt:
    b = baseline[qid]
    h = hyde[qid]
    hd = hyde_docs.get(qid, {})
    hyde_text = hd.get('hyde_doc', h.get('hyde_document', 'N/A'))
    gold = b['gold_doc_ids'][0]
    b_rank = b['retrieved_doc_ids'].index(gold) + 1 if gold in b['retrieved_doc_ids'] else 'not found'
    h_rank = h['retrieved_doc_ids'].index(gold) + 1 if gold in h['retrieved_doc_ids'] else 'not found'
    
    print(f"\n{'─'*60}")
    print(f"Query {qid}: {b['query']}")
    print(f"  Baseline rank: {b_rank} | HyDE rank: {h_rank}")
    print(f"  HyDE doc (full): {hyde_text}")
    print()

# Also check: among both-miss, how many have negation/contradiction language?
both_miss = [qid for qid in baseline 
             if hyde[qid]['hit@10'] == 0 and baseline[qid]['hit@10'] == 0]

print("\n" + "=" * 60)
print(f"PATTERN ANALYSIS")
print("=" * 60)

# Check for negation words in queries
negation_words = ['not', 'no ', 'without', 'lack', 'reduces', 'impaired', 'poorly', 'less', 'decreased', 'inhibits']
negation_queries_miss = []
negation_queries_all = []
for qid in baseline:
    q = baseline[qid]['query'].lower()
    has_neg = any(w in q for w in negation_words)
    if has_neg:
        negation_queries_all.append(qid)
        if baseline[qid]['hit@10'] == 0 and hyde[qid]['hit@10'] == 0:
            negation_queries_miss.append(qid)

print(f"\nQueries with negation/contradiction language: {len(negation_queries_all)}")
print(f"  Of those, both miss@10: {len(negation_queries_miss)} ({100*len(negation_queries_miss)/max(len(negation_queries_all),1):.1f}%)")
print(f"  Overall both miss rate: {100*len(both_miss)/300:.1f}%")

# Check query length patterns
import statistics

all_lengths = [len(baseline[qid]['query'].split()) for qid in baseline]
miss_lengths = [len(baseline[qid]['query'].split()) for qid in both_miss]
hurt_lengths = [len(baseline[qid]['query'].split()) for qid in hyde_hurt]
helped_lengths = [len(baseline[qid]['query'].split()) for qid in 
                  [q for q in baseline if hyde[q]['hit@10']==1 and baseline[q]['hit@10']==0]]

print(f"\nQuery length (words):")
print(f"  All queries: mean={statistics.mean(all_lengths):.1f}, median={statistics.median(all_lengths):.1f}")
print(f"  Both miss:   mean={statistics.mean(miss_lengths):.1f}, median={statistics.median(miss_lengths):.1f}")
if hurt_lengths:
    print(f"  HyDE hurt:   mean={statistics.mean(hurt_lengths):.1f}, median={statistics.median(hurt_lengths):.1f}")
if helped_lengths:
    print(f"  HyDE helped: mean={statistics.mean(helped_lengths):.1f}, median={statistics.median(helped_lengths):.1f}")

# Number of gold docs per query
multi_gold = [qid for qid in baseline if len(baseline[qid]['gold_doc_ids']) > 1]
multi_gold_miss = [qid for qid in both_miss if len(baseline[qid]['gold_doc_ids']) > 1]
print(f"\nMulti-gold-doc queries: {len(multi_gold)}")
print(f"  Of those, both miss: {len(multi_gold_miss)}")


DETAILED FAILURE ANALYSIS: HyDE-HURT CASES

────────────────────────────────────────────────────────────
Query 99: Alizarin forms hydrogen bonds with residues involved in PGAM1 substrate binding.
  Baseline rank: 1 | HyDE rank: 18
  HyDE doc (full): Alizarin, a natural anthraquinone dye, has been shown to interact with various biomolecules through non-covalent interactions, including hydrogen bonding. In the context of phosphoglycerate mutase 1 (PGAM1), a key enzyme in glycolysis, the binding of alizarin to specific residues can influence the enzyme's activity. Studies indicate that alizarin can form hydrogen bonds with amino acid side chains in the active site of PGAM1, particularly with residues such as histidine and aspartate, which are crucial for substrate binding and catalysis. These interactions can stabilize the enzyme-substrate complex,


────────────────────────────────────────────────────────────
Query 415: Female carriers of the Apolipoprotein E4 (APOE4) allele have increas

In [13]:
# 1. Verify rank displacement (26 displaced, 23 promoted)
both_hit = [qid for qid in baseline 
            if hyde[qid]['hit@10'] == 1 and baseline[qid]['hit@10'] == 1]

displaced = []  # baseline top1, hyde not top1
promoted = []   # baseline not top1, hyde top1

for qid in both_hit:
    b1 = baseline[qid].get('hit@1', 0)
    h1 = hyde[qid].get('hit@1', 0)
    if b1 == 1 and h1 == 0:
        displaced.append(qid)
    elif b1 == 0 and h1 == 1:
        promoted.append(qid)

print(f"Displaced from top-1: {len(displaced)}")
print(f"Promoted to top-1:    {len(promoted)}")

# 2. Verify negation analysis (62 queries, 9.7% miss rate)
negation_words = ['not', 'no ', 'without', 'lack', 'reduces', 
                  'impaired', 'poorly', 'less', 'decreased', 'inhibits']

negation_queries = []
negation_both_miss = []
total_both_miss = 0

for qid in baseline:
    q = baseline[qid]['query'].lower()
    both_miss = (baseline[qid]['hit@10'] == 0 and hyde[qid]['hit@10'] == 0)
    if both_miss:
        total_both_miss += 1
    
    has_neg = any(w in q for w in negation_words)
    if has_neg:
        negation_queries.append(qid)
        if both_miss:
            negation_both_miss.append(qid)

print(f"\nTotal queries with negation language: {len(negation_queries)}")
print(f"Of those, both miss@10: {len(negation_both_miss)}")
print(f"Negation miss rate: {100*len(negation_both_miss)/len(negation_queries):.1f}%")
print(f"Overall both miss rate: {100*total_both_miss/300:.1f}%")

# Show the negation queries that missed so reader can verify
print(f"\nNegation queries that both missed:")
for qid in negation_both_miss:
    print(f"  QID {qid}: {baseline[qid]['query']}")


Displaced from top-1: 26
Promoted to top-1:    23

Total queries with negation language: 62
Of those, both miss@10: 6
Negation miss rate: 9.7%
Overall both miss rate: 8.3%

Negation queries that both missed:
  QID 132: Aspirin inhibits the production of PGE2.
  QID 437: Functional consequences of genomic alterations due to Myelodysplastic syndrome (MDS) are poorly understood due to the lack of an animal model.
  QID 502: Healthcare delivery efficiency in crowded delivery centers is impaired by improving structural, logistical, and interpersonal elements.
  QID 674: LDL cholesterol has no involvement in the development of cardiovascular disease.
  QID 1110: Suboptimal nutrition is not predictive of chronic disease
  QID 1316: Transferred UCB T cells acquire a memory-like phenotype in recipients.
